In [70]:
!pip install pycountry

In [71]:
import pandas as pd
import os

In [72]:
SCRIPT_DIR_PATH = os.getcwd()
ROOT_DIR_PATH = os.path.dirname(SCRIPT_DIR_PATH)
DATA_DIR_PATH = os.path.join(ROOT_DIR_PATH, "data")
PROCESSED_DATA_DIR_PATH = os.path.join(DATA_DIR_PATH, "processed_data")

In [73]:
# 1. Define the indicators you want to fetch
# indicators = {
#     "SP.POP.GROW": "pop_growth",
#     "SP.POP.TOTL": "pop_total",
#     "SP.URB.TOTL.IN.ZS": "pop_urban_pct",
#     "SI.POV.DDAY": "poverty_headcount_ratio",
#     "EG.ELC.ACCS.ZS": "access_to_electricity_pct",
#     "EG.USE.ELEC.KH.PC": "electricity_consumption_per_capita_kwh",
#     "EG.USE.PCAP.KG.OE": "energy_use_per_capita_kg_of_oil_equivalent",
#     "EG.USE.COMM.GD.PP.KD": "energy_use_kg_of_oil_equivalent_per_gdp",
#     "EG.ELC.COAL.ZS": "electricity_from_coal_pct",
#     "EG.ELC.NGAS.ZS": "electricity_from_natural_gas_pct",
#     "EG.ELC.RNWX.ZS": "electricity_from_renewables_pct",
#     "EG.FEC.RNEW.ZS": "renewable_energy_consumption_pct",
#     "AG.LND.FRST.ZS": "forest_area_pct",
#     "NV.AGR.TOTL.ZS": "agriculture_value_added_pct",
#     "ER.H2O.FWTL.ZS": "annual_freshwater_withdrawals_pct"
# }


# indicators = {
#     "SP.POP.TOTL": "pop_total",
#     "SP.POP.GROW": "pop_growth",
#     "EG.ELC.RNWX.ZS": "electricity_from_renewables_pct",
#     "EG.FEC.RNEW.ZS": "renewable_energy_consumption_pct",
#     "AG.LND.FRST.ZS": "forest_area_pct",
#     "NY.GDP.MKTP.KD": "gdp_2015_usd",
#     "NY.GDP.MKTP.KD.ZG": "gdp_growth_pct",
#     "NY.GDP.PCAP.KD": "gdp_per_capita_2015_usd",
#     "NY.GDP.PCAP.KD.ZG": "gdp_per_capita_growth_pct",
#     "NE.IMP.GNFS.KD": "imports_2015_usd",
#     "NE.IMP.GNFS.ZS": "imports_pct_of_gdp",
#     "NV.IND.TOTL.KD": "industry_value_added_2015_usd",
#     "NV.IND.TOTL.ZS": "industry_pct_of_gdp",
#     "NV.IND.MANF.KD": "manufacturing_value_added_2015_usd",
#     "NV.IND.MANF.ZS": "manufacturing_pct_of_gdp",
#     "NE.EXP.GNFS.KD": "exports_2015_usd",
#     "NE.EXP.GNFS.ZS": "exports_pct_of_gdp",

# }

# look for transport realted variables.
# look for something related to fossil fuels.
# look for something related to income.
# look for something related to region.

In [74]:
import wbdata
import pandas as pd
import pycountry

# 1) Indicators
indicators = {
    "SP.POP.TOTL": "pop_total",
    "SP.POP.GROW": "pop_growth",
    "EG.ELC.RNWX.ZS": "electricity_from_renewables_pct",
    "EG.FEC.RNEW.ZS": "renewable_energy_consumption_pct",
    "AG.LND.FRST.ZS": "forest_area_pct",
    "NY.GDP.MKTP.PP.KD": "gdp_2021_ppp_intl_usd",
    "NY.GDP.MKTP.KD.ZG": "gdp_growth_pct",
    "NY.GDP.PCAP.PP.KD": "gdp_per_capita_2021_ppp_intl_usd",
    "NY.GDP.PCAP.KD.ZG": "gdp_per_capita_growth_pct",
    "NE.IMP.GNFS.ZS": "imports_pct_of_gdp",
    "NV.IND.TOTL.ZS": "industry_pct_of_gdp",
    "NV.IND.MANF.ZS": "manufacturing_pct_of_gdp",
    "NE.EXP.GNFS.ZS": "exports_pct_of_gdp",
    "IQ.CPA.ENVR.XQ": "cpia_environmental_policy_quality_score",
    "IQ.CPA.PUBS.XQ": "cpia_public_sector_management_quality_score",
    "AG.LND.AGRI.ZS": "agricultural_land_pct",
    "AG.LND.ARBL.ZS": "arable_land_pct",
    "AG.YLD.CREL.KG": "cereal_yield_kg_per_hectare",
    "AG.PRD.CROP.XD": "crop_production_index",
    "AG.PRD.FOOD.XD": "food_production_index",
    "AG.LND.CROP.ZS": "permanent_cropland_pct",
    "AG.CON.FERT.ZS": "fertilizer_consumption_kg_per_ha_arable_land",
    "AG.PRD.LVSK.XD": "livestock_production_index",
    "EG.ELC.RNEW.ZS": "renewable_energy_output_pct",
    "EG.USE.PCAP.KG.OE": "energy_use_kg_of_oil_equivalent_per_capita",
    "EN.URB.MCTY.TL.ZS": "population_in_urban_agglomerations_over_1m_pct",
    "EG.USE.ELEC.KH.PC": "electricity_consumption_kwh_per_capita",
    "EG.USE.COMM.FO.ZS": "fossil_fuel_energy_consumption_pct"
}

import wbdata
import pandas as pd
import pycountry
from time import sleep

dfs = []

for indicator, name in indicators.items():
    print(f"Fetching {indicator} …")

    try:
        tmp = wbdata.get_dataframe(
            {indicator: name},
            country="all",
            freq="Y",
            parse_dates=True,
            skip_cache=True
        ).reset_index()

        dfs.append(tmp)

        # be polite to the API
        sleep(0.3)

    except Exception as e:
        print(f"❌ Failed for {indicator}: {e}")

# merge all indicators
df = dfs[0]
for tmp in dfs[1:]:
    df = df.merge(tmp, on=["country", "date"], how="outer")

df = df.rename(columns={"country": "region", "date": "year"})



Fetching SP.POP.TOTL …


KeyboardInterrupt: 

In [ ]:
# Country lookup
country_df = pd.DataFrame(wbdata.get_countries())[['name','iso2Code']]
country_df.columns = ['region','iso2']

df = df.merge(country_df, on='region', how='left')

def iso2_to_iso3(c2):
    try:
        return pycountry.countries.get(alpha_2=c2).alpha_3
    except:
        return None

df['iso_alpha_3'] = df['iso2'].map(iso2_to_iso3)

df['year'] = df['year'].dt.year
df = df.drop(columns=['iso2', 'region'])


KeyError: 'region'

In [ ]:
# reorder columns
cols = ['iso_alpha_3', 'year'] + [col for col in df.columns if col not in ['iso_alpha_3', 'year']]
df = df[cols]

In [ ]:
df

,iso_alpha_3,year,pop_total,pop_growth,electricity_from_renewables_pct,renewable_energy_consumption_pct,forest_area_pct,gdp_2021_ppp_intl_usd,gdp_growth_pct,gdp_per_capita_2021_ppp_intl_usd,...,arable_land_pct,cereal_yield_kg_per_hectare,crop_production_index,food_production_index,fertilizer_consumption_kg_per_ha_arable_land,renewable_energy_output_pct,energy_use_kg_of_oil_equivalent_per_capita,population_in_urban_agglomerations_over_1m_pct,electricity_consumption_kwh_per_capita,fossil_fuel_energy_consumption_pct
0,AFG,1960,9035043.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.158280,NaN,NaN
1,AFG,1961,9214083.0,1.962239,NaN,NaN,NaN,NaN,NaN,NaN,...,11.728991,1115.1,43.24,41.00,0.143791,NaN,NaN,3.259782,NaN,NaN
2,AFG,1962,9404406.0,2.044523,NaN,NaN,NaN,NaN,NaN,NaN,...,11.805651,1079.0,44.13,41.34,0.142857,NaN,NaN,3.362009,NaN,NaN
3,AFG,1963,9604487.0,2.105208,NaN,NaN,NaN,NaN,NaN,NaN,...,11.882311,985.8,43.02,41.16,0.141935,NaN,NaN,3.465349,NaN,NaN
4,AFG,1964,9814318.0,2.161195,NaN,NaN,NaN,NaN,NaN,NaN,...,11.958972,1082.8,46.91,44.60,0.141026,NaN,NaN,3.570111,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17285,ZWE,2020,15526888.0,1.659353,3.764724,84.1,45.093912,7.030140e+10,-7.816951,4527.719881,...,10.416122,1149.4,127.77,110.34,20.196914,60.785554,373.557246,9.853359,464.806599,0.0
17286,ZWE,2021,15797210.0,1.726011,3.376051,82.4,44.974822,7.625453e+10,8.468017,4827.088694,...,10.408308,1545.5,129.97,120.52,36.868918,72.628388,405.518564,9.760103,540.095371,0.0
17287,ZWE,2022,16069056.0,1.706209,NaN,NaN,44.855732,8.093600e+10,6.139263,5036.761361,...,10.400493,956.9,123.48,121.68,30.780696,NaN,416.586875,9.694036,538.799541,0.0
17288,ZWE,2023,16340822.0,1.677096,NaN,NaN,44.736642,8.526678e+10,5.350869,5218.022665,...,10.392679,743.9,NaN,NaN,26.213262,NaN,NaN,9.657580,NaN,NaN


In [ ]:
df.to_csv(os.path.join(PROCESSED_DATA_DIR_PATH, 'wb_indicators_raw.csv'), index=False)

## Let's explore and clean de df

In [75]:
df = pd.read_csv(os.path.join(PROCESSED_DATA_DIR_PATH, 'wb_indicators_raw.csv'))
df.head()

,iso_alpha_3,year,pop_total,pop_growth,electricity_from_renewables_pct,renewable_energy_consumption_pct,forest_area_pct,gdp_2021_ppp_intl_usd,gdp_growth_pct,gdp_per_capita_2021_ppp_intl_usd,...,arable_land_pct,cereal_yield_kg_per_hectare,crop_production_index,food_production_index,fertilizer_consumption_kg_per_ha_arable_land,renewable_energy_output_pct,energy_use_kg_of_oil_equivalent_per_capita,population_in_urban_agglomerations_over_1m_pct,electricity_consumption_kwh_per_capita,fossil_fuel_energy_consumption_pct
0,AFG,1960,9035043.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.158280,NaN,NaN
1,AFG,1961,9214083.0,1.962239,NaN,NaN,NaN,NaN,NaN,NaN,...,11.728991,1115.1,43.24,41.00,0.143791,NaN,NaN,3.259782,NaN,NaN
2,AFG,1962,9404406.0,2.044523,NaN,NaN,NaN,NaN,NaN,NaN,...,11.805651,1079.0,44.13,41.34,0.142857,NaN,NaN,3.362009,NaN,NaN
3,AFG,1963,9604487.0,2.105208,NaN,NaN,NaN,NaN,NaN,NaN,...,11.882311,985.8,43.02,41.16,0.141935,NaN,NaN,3.465349,NaN,NaN
4,AFG,1964,9814318.0,2.161195,NaN,NaN,NaN,NaN,NaN,NaN,...,11.958972,1082.8,46.91,44.60,0.141026,NaN,NaN,3.570111,NaN,NaN


In [76]:
#Fraction of non-missing observations (overall)
indicator_cols = df.columns.difference(['iso_alpha_3', 'year'])
df_1980 = df[df.year >= 1980]
indicator_coverage = (
    df_1980[indicator_cols]
    .notna()
    .mean()
    .sort_values(ascending=False)
)

indicator_coverage


pop_total                                         0.995405
pop_growth                                        0.995155
gdp_per_capita_growth_pct                         0.910192
gdp_growth_pct                                    0.910192
agricultural_land_pct                             0.904094
arable_land_pct                                   0.889557
food_production_index                             0.821554
crop_production_index                             0.821470
cereal_yield_kg_per_hectare                       0.794987
fertilizer_consumption_kg_per_ha_arable_land      0.794653
industry_pct_of_gdp                               0.782790
exports_pct_of_gdp                                0.748956
forest_area_pct                                   0.720551
manufacturing_pct_of_gdp                          0.710443
gdp_2021_ppp_intl_usd                             0.707185
gdp_per_capita_2021_ppp_intl_usd                  0.707185
renewable_energy_consumption_pct                  0.6878

In [77]:
# Coverage by country (how many countries report it at all)
indicator_country_coverage = (
    df_1980
    .groupby('iso_alpha_3')[indicator_cols]
    .apply(lambda x: x.notna().any())
    .mean()
    .sort_values(ascending=False)
)

indicator_country_coverage


pop_total                                         1.000000
pop_growth                                        1.000000
forest_area_pct                                   0.990698
renewable_energy_consumption_pct                  0.986047
gdp_per_capita_growth_pct                         0.986047
gdp_growth_pct                                    0.986047
agricultural_land_pct                             0.972093
renewable_energy_output_pct                       0.967442
electricity_from_renewables_pct                   0.967442
arable_land_pct                                   0.958140
industry_pct_of_gdp                               0.958140
manufacturing_pct_of_gdp                          0.934884
fertilizer_consumption_kg_per_ha_arable_land      0.930233
gdp_2021_ppp_intl_usd                             0.920930
gdp_per_capita_2021_ppp_intl_usd                  0.920930
food_production_index                             0.906977
crop_production_index                             0.9069

In [78]:
core_indicators = [
    # Demographics / scale
    "pop_total",
    "pop_growth",

    # Economic scale
    "gdp_2021_ppp_intl_usd",
    "gdp_per_capita_2021_ppp_intl_usd",
    "gdp_growth_pct",

    # Energy & emissions intensity (critical)
    "energy_use_kg_of_oil_equivalent_per_capita",
    "electricity_consumption_kwh_per_capita",
    "fossil_fuel_energy_consumption_pct",
]

important_indicators = [
    # Economic structure
    "industry_pct_of_gdp",
    "manufacturing_pct_of_gdp",
    "exports_pct_of_gdp",

    # Energy transition / composition
    "electricity_from_renewables_pct",
    "renewable_energy_consumption_pct",

    # Land use & agriculture (secondary emissions drivers)
    "forest_area_pct",
    "agricultural_land_pct",
    "arable_land_pct",
    "fertilizer_consumption_kg_per_ha_arable_land",

    # Urbanization
    "population_in_urban_agglomerations_over_1m_pct",
]

optional_indicators = list(
    set(indicator_cols)
    - set(core_indicators)
    - set(important_indicators)
)

all_listed = (
    set(core_indicators)
    | set(important_indicators)
    | set(optional_indicators)
)

set(df.columns) - {"iso_alpha_3", "year"} == all_listed


True

In [79]:
indicator_ok = (
    (indicator_coverage >= 0.6) &
    (indicator_country_coverage >= 0.7)
)

kept_indicators = (
    set(core_indicators) |
    set(important_indicators) |
    set(indicator_ok[indicator_ok].index)
)

kept_indicators = list(kept_indicators)
kept_indicators

['pop_growth',
 'food_production_index',
 'pop_total',
 'exports_pct_of_gdp',
 'renewable_energy_output_pct',
 'energy_use_kg_of_oil_equivalent_per_capita',
 'population_in_urban_agglomerations_over_1m_pct',
 'gdp_per_capita_growth_pct',
 'industry_pct_of_gdp',
 'fossil_fuel_energy_consumption_pct',
 'cereal_yield_kg_per_hectare',
 'electricity_consumption_kwh_per_capita',
 'gdp_2021_ppp_intl_usd',
 'gdp_per_capita_2021_ppp_intl_usd',
 'fertilizer_consumption_kg_per_ha_arable_land',
 'crop_production_index',
 'agricultural_land_pct',
 'arable_land_pct',
 'electricity_from_renewables_pct',
 'renewable_energy_consumption_pct',
 'forest_area_pct',
 'gdp_growth_pct',
 'manufacturing_pct_of_gdp']

In [80]:
country_scores = (
    df_1980
    .groupby('iso_alpha_3')[kept_indicators]
    .apply(lambda x: x.notna().mean().mean())
    .sort_values(ascending=False)
)

country_scores

iso_alpha_3
TUR    0.892754
COL    0.892754
NLD    0.892754
FRA    0.892754
SWE    0.892754
         ...   
GIB    0.273430
SSD    0.263768
MCO    0.221256
SXM    0.210628
MAF    0.155556
Length: 215, dtype: float64

In [81]:
target_n = 171
selected_countries = country_scores.head(target_n).index.tolist()


In [82]:
kept_fields = ['iso_alpha_3', 'year'] + kept_indicators

In [ ]:
df_filtered = df_1980[
    df_1980['iso_alpha_3'].isin(selected_countries)
][kept_fields]

# Check final coverage
df_filtered.notna().mean().sort_values()


fossil_fuel_energy_consumption_pct                0.604548
electricity_consumption_kwh_per_capita            0.614685
energy_use_kg_of_oil_equivalent_per_capita        0.626771
renewable_energy_output_pct                       0.644834
electricity_from_renewables_pct                   0.659649
population_in_urban_agglomerations_over_1m_pct    0.684211
renewable_energy_consumption_pct                  0.713840
forest_area_pct                                   0.739571
gdp_per_capita_2021_ppp_intl_usd                  0.763483
gdp_2021_ppp_intl_usd                             0.763483
manufacturing_pct_of_gdp                          0.796751
exports_pct_of_gdp                                0.839636
industry_pct_of_gdp                               0.862248
fertilizer_consumption_kg_per_ha_arable_land      0.894087
cereal_yield_kg_per_hectare                       0.904353
crop_production_index                             0.911501
food_production_index                             0.9115

In [87]:
country_density_filtered = (
    df_filtered
    .groupby(df['iso_alpha_3'])
    .apply(lambda x: x.notna().mean().mean())
)

country_density_filtered.describe()


count    171.000000
mean       0.829214
std        0.067538
min        0.697778
25%        0.764889
50%        0.856889
75%        0.890222
max        0.901333
dtype: float64

In [88]:
df_filtered.head()

,iso_alpha_3,year,pop_growth,food_production_index,pop_total,exports_pct_of_gdp,renewable_energy_output_pct,energy_use_kg_of_oil_equivalent_per_capita,population_in_urban_agglomerations_over_1m_pct,gdp_per_capita_growth_pct,...,gdp_per_capita_2021_ppp_intl_usd,fertilizer_consumption_kg_per_ha_arable_land,crop_production_index,agricultural_land_pct,arable_land_pct,electricity_from_renewables_pct,renewable_energy_consumption_pct,forest_area_pct,gdp_growth_pct,manufacturing_pct_of_gdp
215,ALB,1980,2.047964,41.13,2671997.0,23.957492,NaN,NaN,NaN,NaN,...,NaN,160.170940,48.98,40.802920,21.350365,NaN,NaN,NaN,NaN,NaN
216,ALB,1981,2.002974,42.77,2726056.0,23.810507,NaN,NaN,NaN,3.648649,...,NaN,138.435374,50.89,40.729927,21.459854,NaN,NaN,NaN,5.745635,NaN
217,ALB,1982,2.113272,44.14,2784278.0,20.072918,NaN,NaN,NaN,0.795840,...,NaN,179.966044,53.23,40.656934,21.496350,NaN,NaN,NaN,2.948597,NaN
218,ALB,1983,2.120885,48.08,2843960.0,18.852075,NaN,NaN,NaN,-1.016802,...,NaN,174.363328,59.51,40.510949,21.496350,NaN,NaN,NaN,1.104938,NaN
219,ALB,1984,2.103937,44.62,2904429.0,18.040331,NaN,NaN,NaN,-3.307497,...,NaN,159.592530,54.66,40.620438,21.496350,NaN,NaN,NaN,-1.251597,NaN


In [96]:
df_filtered.to_csv(os.path.join(PROCESSED_DATA_DIR_PATH, 'wb_indicators_filtered.csv'), index=False)